In [1]:
import json

stories = []
with open("stories.jsonl") as f:
    for line in f:
        stories.append(json.loads(line))

print(f"Loaded {len(stories)} stories")
print(f"Example keys: {list(stories[0].keys())}")

Loaded 48 stories
Example keys: ['id', 'pair_id', 'condition', 'domain', 'bias_type', 'prompt', 'story', 'word_count']


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2-9b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

/workspace/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 464/464 [00:58<00:00,  7.88it/s]


Model loaded. VRAM: 8.4 GB


In [3]:
import numpy as np
from tqdm import tqdm

all_activations = []  # will be shape (48, 43, 3584)

for story in tqdm(stories):
    inputs = tokenizer(story["story"], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Mean pool across the sequence dimension
    # hidden_states is a tuple of (43,) tensors each shape (1, seq_len, 3584)
    story_acts = torch.stack([hs[0].mean(dim=0) for hs in outputs.hidden_states])
    # shape: (43, 3584)

    all_activations.append(story_acts.cpu().float().numpy())

all_activations = np.array(all_activations)
print(f"Activations shape: {all_activations.shape}")
print(f"Expected:          (48, 43, 3584)")

100%|██████████| 48/48 [00:09<00:00,  5.05it/s]

Activations shape: (48, 43, 3584)
Expected:          (48, 43, 3584)


In [4]:
np.save("activations.npy", all_activations)                                                                                                
print("Saved activations.npy")      

Saved activations.npy


In [5]:
metadata = [{"id": s["id"], "pair_id": s["pair_id"], "condition": s["condition"], "domain": s["domain"]} for s in stories]

import json
with open("activations_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved activations_metadata.json")
print(f"First entry: {metadata[0]}")
print(f"Last entry:  {metadata[-1]}")

Saved activations_metadata.json
First entry: {'id': 'hiring_01_biased', 'pair_id': 'hiring_01', 'condition': 'biased', 'domain': 'hiring'}
Last entry:  {'id': 'retail_04_neutral', 'pair_id': 'retail_04', 'condition': 'neutral', 'domain': 'retail'}
